# Query-Adaptive Video RAG Pipeline (batch mode, JSON-driven)

Point this at a JSON file of `(video_id, query)` pairs and a Google Drive folder of videos
named by id (`<video_id>.mp4`, `.mov`, etc). For every entry, the matching video is located,
the question is classified into **temporal / spatial / spatio_temporal / counting / audio**,
and only that branch's models run for it — an audio question never loads a vision model, a
counting question never loads Whisper. Per-video results are cached in ChromaDB, so if the
same video shows up across multiple JSON entries its expensive frame/audio processing only
runs once per branch.

**Before running:** Runtime -> Change runtime type -> GPU (T4 or better). You'll be prompted for
a Groq API key (query classification + answer generation) and an OpenAI API key (GPT-4o-mini
captioning for the temporal/spatial branches).

## ⚠️ ABLATION: cache removed

This notebook is identical to the main pipeline **except** that the per-video
ChromaDB chunk cache is disabled. In the original notebook, if a video's chunks for
a given category already exist in ChromaDB, `process_query` reuses them instead of
re-running the (expensive) branch. Here, every single query re-runs its branch's
detection/captioning/ASR from scratch and overwrites whatever was stored before, so
repeated questions about the same video no longer get a free pass on latency.

Everything else — classification, retrieval (dense + BM25), category-restricted
retrieval, and generation — is unchanged from the original pipeline.


## 1. Install dependencies

In [ ]:
# numpy is pinned in the SAME install command as everything else, so pip resolves one
# consistent version up front instead of needing a force-reinstall later (which caused
# "cannot load module more than once per process" when the runtime didn't fully restart).
!pip install -q "numpy<2.2" scenedetect[opencv] transformers accelerate sentence-transformers \
    chromadb rank_bm25 groq openai scikit-learn opencv-python-headless soundfile
!apt-get -qq update && apt-get -qq install -y ffmpeg

**The cell above force-kills the kernel on purpose.** Colab will show "session crashed" for a
second and then reconnect automatically — that's the restart taking effect, not a real crash.

Once it reconnects, go to **Runtime -> Run all** (or just run every cell again from the top,
including this install cell — pip will report everything already satisfied and skip straight to
mounting Drive). You should not see the numpy import error again after this point.

## 2. Upload your video

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# --- EDIT THESE THREE PATHS ---
JSON_PATH = "/content/drive/MyDrive/Internship (1st July to 31st July 2026)/FILTERED_DATASET/MovieChat/Queries/15hard_questions_trimmed.json"   # your queries+ids JSON
VIDEOS_DIR = "/content/drive/MyDrive/Internship (1st July to 31st July 2026)/FILTERED_DATASET/MovieChat/videos"        # folder of videos named by id
OUTPUT_JSON = "/content/drive/MyDrive/path/to/answers_no_cache.json" # where results are written
# --------------------------------

WORKDIR = "/content/work"
os.makedirs(WORKDIR, exist_ok=True)

assert os.path.isfile(JSON_PATH), f"JSON_PATH not found: {JSON_PATH}"
assert os.path.isdir(VIDEOS_DIR), f"VIDEOS_DIR not found: {VIDEOS_DIR}"
print("JSON_PATH:", JSON_PATH)
print("VIDEOS_DIR:", VIDEOS_DIR)
print("OUTPUT_JSON:", OUTPUT_JSON)


## 3. Load (video_id, query) pairs from JSON, locate each video in Drive

Handles a few common JSON shapes automatically:

- a **list** of objects, each with a video-id-ish key (`video_id`/`id`/`vid`/`video`/`video_name`)
  and a query-ish key (`query`/`question`/`prompt`), optionally with an answer-ish key
  (`answer`/`gt`/`ground_truth`/`gt_answer`) for later comparison against the model's output
- a **dict** keyed by video id, where each value is either a list of query strings, a list of
  query objects (as above), or a single query string

If your JSON uses different key names than these, tell me and I'll adjust `_QUERY_KEYS` /
`_ID_KEYS` / `_ANSWER_KEYS` below.

In [ ]:
import json
import glob

_ID_KEYS = ["video_id", "id", "vid", "video", "video_name", "video_file", "video_path"]
_QUERY_KEYS = ["query", "question_content", "prompt", "q", "question"]
_ANSWER_KEYS = ["answer", "gt", "ground_truth", "gt_answer", "expected_answer"]
_CATEGORY_KEYS = ["category", "class_category", "qcategory", "type"]

def _first_key(d, keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None

# query text -> ground-truth category, exactly as written in the input JSON
# (e.g. "Temporal", "Spatial", "Spatiotemporal", "Counting", "Audio"). Populated
# below while loading, used later to report latency broken out by these categories.
GT_CATEGORY_BY_QUERY = {}

def load_video_queries(json_path):
    """Returns dict: video_id -> list of {"query": ..., "expected_answer": ..., "category": ... or None}"""
    with open(json_path) as f:
        data = json.load(f)

    out = {}

    def add(vid, query, answer=None, category=None):
        vid = str(vid).strip()
        query = str(query).strip()
        if not vid or not query:
            return
        out.setdefault(vid, []).append({"query": query, "expected_answer": answer, "category": category})
        if category is not None:
            GT_CATEGORY_BY_QUERY[query] = str(category).strip()

    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict):
                continue
            vid = _first_key(item, _ID_KEYS)
            query = _first_key(item, _QUERY_KEYS)
            answer = _first_key(item, _ANSWER_KEYS)
            category = _first_key(item, _CATEGORY_KEYS)
            if vid is None or query is None:
                print(f"  skipping entry, missing id/query keys: {item}")
                continue
            add(vid, query, answer, category)

    elif isinstance(data, dict):
        for vid, val in data.items():
            if isinstance(val, str):
                add(vid, val)
            elif isinstance(val, list):
                for item in val:
                    if isinstance(item, str):
                        add(vid, item)
                    elif isinstance(item, dict):
                        query = _first_key(item, _QUERY_KEYS)
                        answer = _first_key(item, _ANSWER_KEYS)
                        category = _first_key(item, _CATEGORY_KEYS)
                        if query is None:
                            print(f"  skipping entry for {vid}, no query key: {item}")
                            continue
                        add(vid, query, answer, category)
            elif isinstance(val, dict):
                query = _first_key(val, _QUERY_KEYS)
                answer = _first_key(val, _ANSWER_KEYS)
                category = _first_key(val, _CATEGORY_KEYS)
                if query is not None:
                    add(vid, query, answer, category)
    else:
        raise ValueError(f"Unrecognized JSON top-level type: {type(data)}")

    return out

VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".avi", ".webm", ".m4v"]

def find_video_file(video_id, videos_dir):
    # exact filename match first (id already includes extension)
    direct = os.path.join(videos_dir, video_id)
    if os.path.isfile(direct):
        return direct
    # try id + known extensions
    for ext in VIDEO_EXTS:
        cand = os.path.join(videos_dir, f"{video_id}{ext}")
        if os.path.isfile(cand):
            return cand
    # fall back to glob in case of extra suffixes/case differences
    matches = glob.glob(os.path.join(videos_dir, f"{video_id}.*"))
    if matches:
        return matches[0]
    matches = glob.glob(os.path.join(videos_dir, f"{video_id}*"))
    if matches:
        return matches[0]
    return None

VIDEO_QUERIES = load_video_queries(JSON_PATH)
print(f"Loaded {sum(len(v) for v in VIDEO_QUERIES.values())} queries across {len(VIDEO_QUERIES)} video id(s)")
print(f"Ground-truth categories found for {len(GT_CATEGORY_BY_QUERY)} of them")

# Resolve video files up front so missing videos are reported before any processing starts
RESOLVED = {}
MISSING = []
for vid in VIDEO_QUERIES:
    path = find_video_file(vid, VIDEOS_DIR)
    if path is None:
        MISSING.append(vid)
    else:
        RESOLVED[vid] = path

print(f"Resolved {len(RESOLVED)} video(s), missing {len(MISSING)}")
if MISSING:
    print("Missing video ids (no matching file in VIDEOS_DIR):")
    for vid in MISSING:
        print(f"  - {vid}")


## 4. API keys

In [ ]:
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (GPT-4o-mini captioning): ")


## 5. Device + lazy model cache\n\nModels only load the first time a branch actually needs them, so unused branches never touch the GPU.

In [ ]:
import torch, numpy as np
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

_MODEL_CACHE = {}

def get_model(name, loader):
    if name not in _MODEL_CACHE:
        print(f"Loading {name} ...")
        _MODEL_CACHE[name] = loader()
    return _MODEL_CACHE[name]


## 6. Query classifier — Llama-4-Maverick (Groq)

In [ ]:
from groq import Groq
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Check Groq's current model catalog (console.groq.com/docs/models) for the exact model id.
CLASSIFIER_MODEL = "openai/gpt-oss-120b"
GENERATION_MODEL = "openai/gpt-oss-120b"

CLASSIFY_PROMPT = """Classify the question about a video into exactly one category:
- temporal: ordering, sequence, before/after, when something happened
- spatial: layout, position, where something is at a single moment
- spatio_temporal: how something moved or changed position over time
- counting: how many of something
- audio: about speech, dialogue, or non-speech sounds

Reply with only the category name, nothing else.

Question: {query}
Category:"""

VALID_CATEGORIES = {"temporal", "spatial", "spatio_temporal", "counting", "audio"}

def classify_query(query: str) -> str:
    resp = groq_client.chat.completions.create(
        model=CLASSIFIER_MODEL,
        messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(query=query)}],
        temperature=0,
        max_tokens=200,
        reasoning_effort="low",
    )
    content = resp.choices[0].message.content
    label = content.strip().lower() if content else ""
    return label if label in VALID_CATEGORIES else "spatial"  # safe fallback

## 7. Shared ffmpeg utilities

In [ ]:
def extract_audio(video_path, out_wav):
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-ar", "16000", "-ac", "1", out_wav],
        check=True, capture_output=True,
    )
    return out_wav

def extract_frames(video_path, out_dir, fps=1):
    os.makedirs(out_dir, exist_ok=True)
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vf", f"fps={fps}", f"{out_dir}/frame_%05d.jpg"],
        check=True, capture_output=True,
    )
    return sorted(glob.glob(f"{out_dir}/*.jpg"))

def caption_frame_gpt4o_mini(image_path, instruction="Describe this scene in one sentence."):
    import base64
    from openai import OpenAI
    client = get_model("openai_client", lambda: OpenAI())
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ],
        }],
        max_tokens=100,
    )
    return resp.choices[0].message.content.strip()


## 8. Temporal branch — PySceneDetect + GPT-4o-mini

In [ ]:
from scenedetect import detect, ContentDetector

def run_temporal_branch(video_path, query, workdir):
    scenes = detect(video_path, ContentDetector())
    frame_dir = os.path.join(workdir, "temporal_frames")
    os.makedirs(frame_dir, exist_ok=True)
    chunks = []
    for i, (start, end) in enumerate(scenes):
        mid_sec = (start.get_seconds() + end.get_seconds()) / 2
        frame_path = os.path.join(frame_dir, f"scene_{i:03d}.jpg")
        subprocess.run(
            ["ffmpeg", "-y", "-ss", str(mid_sec), "-i", video_path, "-frames:v", "1", frame_path],
            check=True, capture_output=True,
        )
        caption = caption_frame_gpt4o_mini(frame_path)
        chunks.append({
            "text": caption,
            "category": "temporal",
            "start_ts": start.get_seconds(),
            "end_ts": end.get_seconds(),
            "order": i,
        })
    return chunks


## 9. Spatial branch — CLIP keyframes + GPT-4o-mini + GroundingDINO

In [ ]:
from transformers import CLIPModel, CLIPProcessor
from sklearn.cluster import KMeans

def load_clip():
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    return model, processor

def select_keyframes(frame_paths, k=8):
    model, processor = get_model("clip", load_clip)
    embs = []
    for p in frame_paths:
        img = Image.open(p).convert("RGB")
        inputs = processor(images=img, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            emb = model.get_image_features(**inputs)
        # transformers >=5 returns a BaseModelOutputWithPooling instead of a plain tensor;
        # the projected image embedding lives in .pooler_output
        if hasattr(emb, "pooler_output"):
            emb = emb.pooler_output
        embs.append(emb.cpu().numpy().flatten())
    embs = np.stack(embs)
    k = min(k, len(frame_paths))
    labels = KMeans(n_clusters=k, n_init=5, random_state=0).fit_predict(embs)
    keyframes = []
    for c in range(k):
        idxs = np.where(labels == c)[0]
        center = embs[idxs].mean(axis=0)
        best = idxs[np.argmin(np.linalg.norm(embs[idxs] - center, axis=1))]
        keyframes.append(frame_paths[best])
    return sorted(keyframes)

def load_grounding_dino():
    from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
    processor = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
    model = AutoModelForZeroShotObjectDetection.from_pretrained("IDEA-Research/grounding-dino-tiny").to(DEVICE)
    return model, processor

def ground_objects(image_path, text_query):
    model, processor = get_model("grounding_dino", load_grounding_dino)
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, text=text_query, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_grounded_object_detection(
    outputs, threshold=0.3,
    target_sizes=[img.size[::-1]],
    )[0]
    label_key = "text_labels" if "text_labels" in results else "labels"
    return [{"label": l, "box": b.tolist()} for l, b in zip(results[label_key], results["boxes"])]

def run_spatial_branch(video_path, query, workdir):
    frame_dir = os.path.join(workdir, "spatial_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=1)
    keyframes = select_keyframes(frame_paths, k=8)
    chunks = []
    for i, fp in enumerate(keyframes):
        caption = caption_frame_gpt4o_mini(fp, "Describe the layout and position of objects in this frame.")
        objects = ground_objects(fp, query)
        obj_str = ", ".join(o["label"] for o in objects) if objects else "none detected"
        chunks.append({
            "text": f"{caption} Objects grounded: {obj_str}.",
            "category": "spatial",
            "frame_id": i,
        })
    return chunks


## 10. Spatio-Temporal branch — RT-DETR + BoT-SORT

In [ ]:
import torch
from pathlib import Path

REID_MODEL_PATH = Path("osnet_x0_25_msmt17.pt")
REID_MODEL_URL = "https://github.com/mikel-brostrom/Yolov5_StrongSORT_OSNet/releases/download/v.1.2.7/osnet_x0_25_msmt17.pt"

def load_rtdetr():
    from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
    processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd_coco_o365")
    model = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd_coco_o365").to(DEVICE)
    return model, processor

def detect_objects_rtdetr(image_path, threshold=0.4):
    model, processor = get_model("rtdetr", load_rtdetr)
    img = Image.open(image_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_object_detection(
        outputs, target_sizes=torch.tensor([img.size[::-1]]), threshold=threshold,
    )[0]
    return np.column_stack([
        results["boxes"].cpu().numpy(),
        results["scores"].cpu().numpy(),
        results["labels"].cpu().numpy(),
    ])  # [x1, y1, x2, y2, score, class_id]

from scipy.optimize import linear_sum_assignment

def iou_batch(a, b):
    a, b = np.expand_dims(a, 1), np.expand_dims(b, 0)
    xx1 = np.maximum(a[..., 0], b[..., 0])
    yy1 = np.maximum(a[..., 1], b[..., 1])
    xx2 = np.minimum(a[..., 2], b[..., 2])
    yy2 = np.minimum(a[..., 3], b[..., 3])
    inter = np.maximum(0., xx2 - xx1) * np.maximum(0., yy2 - yy1)
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / np.clip(area_a + area_b - inter, 1e-6, None)

class _Track:
    _next_id = 1
    def __init__(self, box, cls, conf):
        self.id = _Track._next_id; _Track._next_id += 1
        self.box = np.array(box, dtype=float)
        self.velocity = np.zeros(4)
        self.cls, self.conf = cls, conf
        self.hits, self.time_since_update = 1, 0

    def predict(self):
        self.box = self.box + self.velocity
        self.time_since_update += 1
        return self.box

    def update(self, box, conf):
        box = np.array(box, dtype=float)
        self.velocity = 0.5 * self.velocity + 0.5 * (box - self.box)
        self.box, self.conf = box, conf
        self.hits += 1
        self.time_since_update = 0

class SimpleSort:
    def __init__(self, iou_threshold=0.3, max_age=5, min_hits=1):
        self.iou_threshold, self.max_age, self.min_hits = iou_threshold, max_age, min_hits
        self.tracks = []

    def update(self, dets, img=None):
        dets = np.asarray(dets, dtype=float) if len(dets) else np.empty((0, 6))
        for t in self.tracks:
            t.predict()

        if not self.tracks:
            matches, unmatched_dets, unmatched_tracks = [], list(range(len(dets))), []
        elif len(dets) == 0:
            matches, unmatched_dets, unmatched_tracks = [], [], list(range(len(self.tracks)))
        else:
            track_boxes = np.array([t.box for t in self.tracks])
            iou_mat = iou_batch(dets[:, :4], track_boxes)
            for i, d in enumerate(dets):
                for j, t in enumerate(self.tracks):
                    if d[5] != t.cls:
                        iou_mat[i, j] = 0
            row_ind, col_ind = linear_sum_assignment(-iou_mat)
            matches, unmatched_dets, unmatched_tracks = [], list(range(len(dets))), list(range(len(self.tracks)))
            for r, c in zip(row_ind, col_ind):
                if iou_mat[r, c] >= self.iou_threshold:
                    matches.append((r, c))
                    unmatched_dets.remove(r)
                    unmatched_tracks.remove(c)

        for d_idx, t_idx in matches:
            self.tracks[t_idx].update(dets[d_idx, :4], dets[d_idx, 4])
        for d_idx in unmatched_dets:
            self.tracks.append(_Track(dets[d_idx, :4], dets[d_idx, 5], dets[d_idx, 4]))
        self.tracks = [t for t in self.tracks if t.time_since_update <= self.max_age]

        rows = [
            [*t.box, t.id, t.cls, t.conf]
            for t in self.tracks if t.time_since_update == 0 and t.hits >= self.min_hits
        ]
        return np.array(rows) if rows else np.empty((0, 7))

def load_tracker():
    return SimpleSort(iou_threshold=0.3, max_age=5, min_hits=1)
def run_spatiotemporal_branch(video_path, query, workdir, fps=2):
    frame_dir = os.path.join(workdir, "st_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=fps)
    tracker = get_model("bot_sort_st", load_tracker)
    tracks = {}
    for idx, fp in enumerate(frame_paths):
        dets = detect_objects_rtdetr(fp)
        img = np.array(Image.open(fp).convert("RGB"))
        tracked = tracker.update(dets, img) if len(dets) else []
        for row in tracked:
            x1, y1, x2, y2, tid, cls, conf = row[:7]
            tracks.setdefault(int(tid), []).append((idx, (x1, y1, x2, y2), int(cls)))
    chunks = []
    for tid, path in tracks.items():
        start_idx, start_box, cls = path[0]
        end_idx, end_box, _ = path[-1]
        text = (f"Object class {cls} (track {tid}) moved from {region_of(start_box)} to "
                f"{region_of(end_box)} between t={start_idx/fps:.1f}s and t={end_idx/fps:.1f}s.")
        chunks.append(
            {"text": text, "category": "spatio_temporal",
            "track_id": tid, "start_ts": start_idx / fps, "end_ts": end_idx / fps,
        })
    return chunks

## 11. Counting branch — RT-DETR + BoT-SORT dedup

In [ ]:
def run_counting_branch(video_path, query, workdir, fps=1):
    frame_dir = os.path.join(workdir, "count_frames")
    frame_paths = extract_frames(video_path, frame_dir, fps=fps)
    tracker = get_model("bot_sort_count", load_tracker)
    class_track_ids = {}
    evidence = {}
    for fp in frame_paths:
        dets = detect_objects_rtdetr(fp)
        img = np.array(Image.open(fp).convert("RGB"))
        tracked = tracker.update(dets, img) if len(dets) else []
        for row in tracked:
            x1, y1, x2, y2, tid, cls, conf = row[:7]
            class_track_ids.setdefault(int(cls), set()).add(int(tid))
            evidence.setdefault(int(cls), fp)
    chunks = []
    for cls, ids in class_track_ids.items():
        chunks.append({
            "text": f"{len(ids)} unique instance(s) of class {cls} observed across the video.",
            "category": "counting",
            "class_id": cls, "count": len(ids), "evidence_frame": evidence[cls],
        })
    return chunks


## 12. Audio branch — Whisper large-v3-turbo + AST

In [ ]:
from transformers import pipeline as hf_pipeline

def load_whisper():
    return hf_pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-large-v3-turbo",
        device=0 if DEVICE == "cuda" else -1,
        chunk_length_s=30,
        return_timestamps=True,
    )

def load_ast():
    return hf_pipeline(
        "audio-classification",
        model="MIT/ast-finetuned-audioset-10-10-0.4593",
        device=0 if DEVICE == "cuda" else -1,
    )

def run_audio_branch(video_path, query, workdir):
    wav_path = os.path.join(workdir, "audio.wav")
    extract_audio(video_path, wav_path)

    asr = get_model("whisper", load_whisper)
    transcript = asr(wav_path)
    segments = transcript.get("chunks") or [{"text": transcript["text"], "timestamp": (0, None)}]

    ast = get_model("ast", load_ast)
    tags = ast(wav_path, top_k=5)
    tag_str = ", ".join(t["label"] for t in tags)

    chunks = []
    for seg in segments:
        start, end = seg["timestamp"]
        text = seg["text"].strip()
        if not text:
            continue
        chunks.append({
            "text": text, "category": "audio",
            "start_ts": start, "end_ts": end, "event_tags": tag_str,
        })
    return chunks


## 13. Context assembly — BGE embeddings in ChromaDB, hybrid retrieval

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi

def load_bge():
    return SentenceTransformer("BAAI/bge-large-en-v1.5", device=DEVICE)

chroma_client = chromadb.PersistentClient(path=os.path.join(WORKDIR, "chroma"))

def get_collection(video_id):
    return chroma_client.get_or_create_collection(name=f"video_{video_id}")

def _sanitize_metadata(md):
    # Chroma metadata values must be str/int/float/bool -- no None, list, or dict.
    # e.g. Whisper leaves end_ts=None when it cannot predict a segment's ending timestamp.
    clean = {}
    for k, v in md.items():
        if v is None:
            continue  # drop rather than coerce, so missing timestamps don't become the string "None"
        if isinstance(v, (str, int, float, bool)):
            clean[k] = v
        else:
            clean[k] = str(v)
    return clean

def add_chunks(video_id, chunks):
    if not chunks:
        return
    bge = get_model("bge", load_bge)
    coll = get_collection(video_id)
    texts = [c["text"] for c in chunks]
    embeddings = bge.encode(texts, normalize_embeddings=True).tolist()
    ids = [f"{video_id}_{c['category']}_{i}" for i, c in enumerate(chunks)]
    metadatas = [_sanitize_metadata({k: v for k, v in c.items() if k != "text"}) for c in chunks]
    coll.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)

def hybrid_retrieve(video_id, query, category, top_k=6):
    bge = get_model("bge", load_bge)
    coll = get_collection(video_id)
    q_emb = bge.encode([query], normalize_embeddings=True).tolist()
    dense = coll.query(query_embeddings=q_emb, n_results=top_k, where={"category": category})
    docs, metas = dense["documents"][0], dense["metadatas"][0]

    if len(docs) > 1:
        tokenized = [d.lower().split() for d in docs]
        bm25 = BM25Okapi(tokenized)
        scores = bm25.get_scores(query.lower().split())
        order = np.argsort(scores)[::-1]
        docs = [docs[i] for i in order]
        metas = [metas[i] for i in order]

    return list(zip(docs, metas))


## 14. Answer generation — Llama-4-Maverick (Groq), category-specific prompts

In [ ]:
PROMPT_TEMPLATES = {
    "temporal": "Use the timestamped scene descriptions below, in order, to answer the question about sequence or timing.",
    "spatial": "Use the frame descriptions below, which describe layout and object positions, to answer the question.",
    "spatio_temporal": "Use the object trajectory descriptions below to answer the question about movement over time.",
    "counting": "Use the count records below. Report the exact numbers if available or else estimate.",
    "audio": "Use the transcript and audio event tags below to answer the question about speech or sound.",
}

def generate_answer(query, category, retrieved):
    context = "\n".join(f"- {doc}" for doc, _ in retrieved) or "(no context retrieved)"
    instruction = PROMPT_TEMPLATES.get(category, "Use the context below to answer the question.")
    prompt = f"{instruction}\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    resp = groq_client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1024,
        reasoning_effort="low",
    )
    content = resp.choices[0].message.content
    return content.strip() if content else "(model returned no content — try increasing max_tokens)"

## 15. Orchestration\n\nOnly the classified branch's function is ever called — the other four never run.

In [ ]:
import re
import traceback

BRANCH_FUNCS = {
    "temporal": run_temporal_branch,
    "spatial": run_spatial_branch,
    "spatio_temporal": run_spatiotemporal_branch,
    "counting": run_counting_branch,
    "audio": run_audio_branch,
}

def process_query(video_path, query):
    try:
        raw_id = os.path.splitext(os.path.basename(video_path))[0]
        video_id = re.sub(r"[^a-zA-Z0-9._-]+", "_", raw_id).strip("_-.")[:200]
        if len(video_id) < 3:
            video_id = f"video_{abs(hash(raw_id)) % 10**8}"
        category = classify_query(query)
        print(f"[{video_id}] '{query}' -> {category}")

        video_workdir = os.path.join(WORKDIR, video_id)
        os.makedirs(video_workdir, exist_ok=True)

        coll = get_collection(video_id)

        # --- CACHE ABLATION ---
        # The original pipeline checked `coll.get(where={"category": category})` and
        # skipped re-running the branch if chunks already existed. That cache is
        # removed here: we always re-run the branch, and wipe any previously stored
        # chunks for this (video, category) first so ChromaDB doesn't accumulate
        # duplicates across repeated queries on the same video.
        existing = coll.get(where={"category": category}) if coll.count() else {"ids": []}
        if existing["ids"]:
            coll.delete(ids=existing["ids"])

        print(f"  running {category} branch (cache disabled -- always re-extracting)")
        chunks = BRANCH_FUNCS[category](video_path, query, video_workdir)
        print(f"  {category} branch produced {len(chunks)} chunk(s)")
        add_chunks(video_id, chunks)

        retrieved = hybrid_retrieve(video_id, query, category)
        print(f"  retrieved {len(retrieved)} chunk(s) for generation")
        answer = generate_answer(query, category, retrieved)
        return {"video": video_id, "query": query, "category": category, "answer": answer}
    except Exception:
        print(f"--- process_query failed on: {query!r} ---")
        traceback.print_exc()
        raise



In [ ]:
_MODEL_CACHE.pop("bot_sort_st", None)
_MODEL_CACHE.pop("bot_sort_count", None)
print("cleared stale tracker cache")

## 15b. Latency instrumentation (per query, tagged by the JSON's category)

Wraps `process_query` so every call's wall-clock time is recorded, tagged with the
**category as given in the input JSON** (`category`/`class_category`/`type` field — e.g.
`Temporal`, `Spatial`, `Spatiotemporal`, `Counting`, `Audio`), not the classifier's own
prediction. The classifier's predicted category is still recorded alongside it as
`predicted_category`, so you can spot mismatches. **Run this cell once**, after
`process_query` is defined (section 15) and before the batch loop (section 16) —
re-running it would wrap an already-wrapped function and double-count time.


In [ ]:
import time

_ORIGINAL_process_query = process_query
# one entry per (video, query) call: video, query, category (from the input JSON),
# predicted_category (from the classifier), latency_sec
LATENCY_LOG = []

def process_query(video_path, query):
    start = time.perf_counter()
    gt_category = GT_CATEGORY_BY_QUERY.get(query, "unknown")
    try:
        result = _ORIGINAL_process_query(video_path, query)
        elapsed = time.perf_counter() - start
        predicted_category = (result or {}).get("category", "unknown")
        LATENCY_LOG.append({
            "video": (result or {}).get("video", video_path),
            "query": query,
            "category": gt_category,
            "predicted_category": predicted_category,
            "latency_sec": elapsed,
        })
        return result
    except Exception:
        elapsed = time.perf_counter() - start
        LATENCY_LOG.append({
            "video": video_path,
            "query": query,
            "category": gt_category,
            "predicted_category": "error",
            "latency_sec": elapsed,
        })
        raise


## 16. Run every (video, query) pair from the JSON and save results

Videos are processed in order of first appearance; each video's branch chunks are cached in
ChromaDB (see cell 31 `process_query`), so repeat questions about the same video reuse cached
chunks instead of re-running detection/ASR/etc.

In [ ]:
import subprocess
import glob
import json
results = []
total = sum(len(v) for v in VIDEO_QUERIES.values())
done = 0

for vid, items in VIDEO_QUERIES.items():
    video_path = RESOLVED.get(vid)
    if video_path is None:
        for item in items:
            done += 1
            print(f"[{done}/{total}] SKIP missing video '{vid}': {item['query']!r}")
            results.append({
                "video": vid,
                "query": item["query"],
                "expected_answer": item.get("expected_answer"),
                "expected_category": item.get("category"),
                "error": "video file not found in VIDEOS_DIR",
            })
        continue

    for item in items:
        done += 1
        query = item["query"]
        print(f"[{done}/{total}] video={vid!r} query={query!r}")
        try:
            result = process_query(video_path, query)
            if result is None:
                result = {"video": vid, "query": query, "error": "process_query returned no result"}
            result["expected_answer"] = item.get("expected_answer")
            result["expected_category"] = item.get("category")
            results.append(result)
        except Exception as e:
            print(f"  Error on '{query}' for video '{vid}': {e}")
            results.append({
                "video": vid,
                "query": query,
                "expected_answer": item.get("expected_answer"),
                "expected_category": item.get("category"),
                "error": str(e),
            })



for r in results:
    if not r:
        print("(empty result, skipped)\n")
        continue
    print(f"[{r.get('video')}] Q: {r.get('query')}")
    print(f"  [{r.get('category', 'error')}] {r.get('answer', r.get('error'))}")
    if r.get('expected_answer') is not None:
        print(f"  expected: {r.get('expected_answer')}")
    print()


## 17. Per-category latency summary

Aggregates `LATENCY_LOG` into count / total / average / min / max latency per query
category, attaches each query's own latency to its result row, and writes everything —
answers plus latency stats — to `OUTPUT_JSON`.

*(Note: the batch loop above builds `results` in memory but never actually calls
`json.dump` to `OUTPUT_JSON` — this cell is what actually writes the file, so run it
before the download cell below.)*

In [ ]:
import json
from collections import defaultdict

# attach per-query latency + predicted category to each result row (match by video + query)
_lat_by_key = {(l["video"], l["query"]): l for l in LATENCY_LOG}
for r in results:
    key = (r.get("video"), r.get("query"))
    if key in _lat_by_key:
        l = _lat_by_key[key]
        r["latency_sec"] = round(l["latency_sec"], 4)
        r["predicted_category"] = l["predicted_category"]

# aggregate by the category given in the input JSON (falls back to "unknown" if a query
# had no category field / wasn't found in GT_CATEGORY_BY_QUERY)
_by_cat = defaultdict(list)
for l in LATENCY_LOG:
    _by_cat[l["category"]].append(l["latency_sec"])

latency_by_category = {}
for cat, times in _by_cat.items():
    latency_by_category[cat] = {
        "count": len(times),
        "total_latency_sec": round(sum(times), 4),
        "avg_latency_sec": round(sum(times) / len(times), 4),
        "min_latency_sec": round(min(times), 4),
        "max_latency_sec": round(max(times), 4),
    }

print("Latency by category (from input JSON):")
for cat, stats in latency_by_category.items():
    print(f"  {cat:16s} n={stats['count']:3d}  avg={stats['avg_latency_sec']:.2f}s  "
          f"min={stats['min_latency_sec']:.2f}s  max={stats['max_latency_sec']:.2f}s  "
          f"total={stats['total_latency_sec']:.2f}s")

# flag any queries where the classifier's predicted category didn't match the JSON's category
_mismatches = [l for l in LATENCY_LOG if l["predicted_category"] not in ("error",)
               and l["category"] != "unknown"
               and l["predicted_category"].replace("_", "").lower() != l["category"].replace("_", "").lower()]
if _mismatches:
    print(f"\n{len(_mismatches)} quer(y/ies) where the classifier's prediction differed from the JSON's category:")
    for l in _mismatches:
        print(f"  '{l['query'][:60]}...' -> json={l['category']!r} vs predicted={l['predicted_category']!r}")

output_payload = {
    "results": results,
    "latency_by_category": latency_by_category,
}

os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)
with open(OUTPUT_JSON, "w") as f:
    json.dump(output_payload, f, indent=2)

print(f"\nSaved {len(results)} result(s) and latency stats to {OUTPUT_JSON}")


## 18. (Optional) also download the results file locally

Results are already saved to OUTPUT_JSON on Drive (see above). This just also pops a local
download if you want a copy outside Drive.

In [ ]:
from google.colab import files as colab_files
colab_files.download(OUTPUT_JSON)
